In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from fintl.cli.etl import run

In [ ]:
run()

In [ ]:
import re


def detect_date_format(date_string: str) -> str:
    # Define regex patterns for different date formats
    patterns = [
        (r"^\d{2}\.\d{2}\.\d{2}$", "%d.%m.%y"),  # 22.11.24
        (r"^\d{2}\.\d{2}\.\d{4}$", "%d.%m.%Y"),  # 22.11.2024
        (r"^\d{2}/\d{2}/\d{2}$", "%d/%m/%y"),  # 22/11/24
        (r"^\d{2}/\d{2}/\d{4}$", "%d/%m/%Y"),  # 22/11/2024
        (r"^\d{2}-\d{2}-\d{2}$", "%d-%m-%y"),  # 22-11-24
        (r"^\d{2}-\d{2}-\d{4}$", "%d-%m-%Y"),  # 22-11-2024
        (r"^\d{4}-\d{2}-\d{2}$", "%Y-%m-%d"),  # 2024-11-24
    ]

    for pattern, format_string in patterns:
        if re.match(pattern, date_string):
            return format_string

    raise ValueError("Unable to detect date format. Please check your input data.")


date_strings = [
    "22.11.24",
    "23.11.24",
    "24.11.24",
    "22.11.2024",
    "22/11/24",
    "22/11/2024",
    "22-11-2024",
    "2024-11-22",
]
for ds in date_strings:
    print(ds, detect_date_format(ds))

In [ ]:
from pathlib import Path

file_path = Path("")
file_path.exists()

In [ ]:
from fintl.accounts_etl.utils import find_line_with_pattern

In [ ]:
encoding = "UTF-8-SIG"  # "utf-8"

In [ ]:
with file_path.open("r", encoding=encoding) as f:
    lines = f.readlines()

In [ ]:
transaction_pattern: str = '^("?Buchungsdatum)'  # start of transactions

In [ ]:
ix_start_transactions, transactions_header = find_line_with_pattern(
    lines, pattern=transaction_pattern
)
ix_start_transactions, transactions_header

In [ ]:
import polars as pl

In [ ]:
# ix_start_transactions = 4
is_empty_1st_line = False
separator = ";"

schema = {
    "Buchungsdatum": pl.Utf8,
    "Wertstellung": pl.Utf8,
    "Status": pl.Utf8,
    "Zahlungspflichtige*r": pl.Utf8,
    "Zahlungsempfänger*in": pl.Utf8,
    "Verwendungszweck": pl.Utf8,
    "Umsatztyp": pl.Utf8,
    "IBAN": pl.Utf8,
    "Betrag": pl.Utf8,
    "Gläubiger-ID": pl.Utf8,
    "Mandatsreferenz": pl.Utf8,
    "Kundenreferenz": pl.Utf8,
}

In [ ]:
transactions = pl.read_csv(
    file_path,
    skip_rows=ix_start_transactions - 1 if is_empty_1st_line else ix_start_transactions,
    separator=separator,
    truncate_ragged_lines=True,
    encoding=encoding,
    schema=schema,
)

In [ ]:
transactions.head()

In [ ]:
date_cols: list = ["Buchungsdatum"]

In [ ]:
transactions[date_cols[0]].to_list()

In [ ]:
[detect_date_format(transactions[col].to_list()[0]) for col in date_cols]

In [ ]:
date_format: str = "%d.%m.%y"
transactions.with_columns(
    [pl.col(col).str.to_date(date_format) for col in date_cols],
)